# R3: Empirical p(True), verbalized confidence 2S and model list

**Goal:** Build the reflexive methods that need **no logits**, only the generated text, so they also work
with API models. Two methods:

- **Empirical p(True):** ask the model 10 times whether the answer is True or False; confidence = share of "True".
- **Verbalized confidence 2S:** after the answer, ask "How confident are you? (0-100)"; confidence = the number.

Questions: Do they separate correct from wrong answers (sanity test, 3 model sizes)? How do they do on
50 real TriviaQA questions (Qwen2.5-0.5B)? What do they cost compared with the logit-based methods?

This notebook **reads saved results**; the model runs are done by the scripts below, so *Restart & Run All*
takes seconds.

## Setup

Reproduce the results (from the repo root):

```
uv run python experiments/llm-probly/scripts/model_list.py | tee experiments/llm-probly/results/model_list.txt
uv run python experiments/llm-probly/scripts/sanity_r3.py        # sanity test, 3 models
uv run python experiments/llm-probly/scripts/r3_real_data.py     # 50 TriviaQA questions
uv run python experiments/llm-probly/scripts/r3_cost_table.py    # cost table
```

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# experiments/llm-probly, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir() and (p / "scripts").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

import r3_methods as r3  # the code of both methods (scripts/r3_methods.py)
from common.sanity import SANITY  # R2's shared sanity questions

rows = [json.loads(line) for line in (ROOT / "data/qwen0.5b/triviaqa_50.jsonl").read_text().splitlines() if line.strip()]
ids = [r["id"] for r in rows]
correct = np.array([r["correct"] for r in rows], dtype=bool)
print(f"{len(rows)} questions, {correct.sum()} greedy answers correct ({correct.mean():.0%})")

50 questions, 12 greedy answers correct (24%)


## Implementation

Both methods live in `scripts/r3_methods.py` and are shared by the sanity test and the 50-question run.

**Empirical p(True)** uses LM-Polygraph's p(True) prompt, sampled 10 times (temperature 1.0, max 3 new tokens).
A reply counts as True if it starts with "true" or is the option letter A / (A) / A). The plan's looser rule
(`startswith("a")`) also counted words that merely start with "a" ("Answer: ...", "Absolutely ..."), so it was
made stricter: a lone "A" still counts.
If none of the 10 replies can be read as True or False, the question is marked `failed`.

In [2]:
print(r3.PT_PROMPT.format(q="What is the capital of France?", a="Berlin"))
print()
for reply in ["(A) True", "(B) False", "True", "A)", "Answer: True", "The answer is"]:
    verdict = "True" if r3.is_true(reply) else "False" if r3.is_false(reply) else "unreadable"
    print(f"{reply!r:<18} -> {verdict}")

Question: What is the capital of France?
Possible answer: Berlin
Is the possible answer:
 (A) True
 (B) False
The possible answer is:

'(A) True'         -> True
'(B) False'        -> False
'True'             -> True
'A)'               -> True
'Answer: True'     -> unreadable
'The answer is'    -> unreadable


**Verbalized confidence 2S** is a two-stage chat: question -> answer -> "How confident are you that your answer
is correct? Reply with only a number between 0 and 100." (greedy, max 8 new tokens). The first number in the
reply is read. A decimal number <= 1 (e.g. "0.99") is already a share; before this fix it was read as 0.99 %.
No number -> confidence 0, `failed` = true, so uncertainty = 1.0 as the score contract requires.

In [3]:
print(r3.ASK[100])
print()
for reply in ["95", "I am 90% confident that", "0.99", "eight"]:
    print(f"{reply!r:<28} -> {r3.parse_confidence(reply, 100)}")

How confident are you that your answer is correct? Reply with only a number between 0 and 100.

'95'                         -> 0.95
'I am 90% confident that'    -> 0.9
'0.99'                       -> 0.99
'eight'                      -> None


## Model list

Memory, load time and time for 10 answers (temperature 0.7, max 32 new tokens) per model, measured with
`scripts/model_list.py`.

In [4]:
model_list = ROOT / "results/model_list.txt"
if model_list.exists():
    table = [line for line in model_list.read_text().splitlines() if line.startswith("|")]
    display(Markdown("\n".join(table)))
else:
    print("results/model_list.txt missing: run scripts/model_list.py (see Setup)")

| Model | Parametre | Bellek (fp32) | Yukleme | 10 cevap CPU | 10 cevap MPS |
|---|---|---|---|---|---|
| HuggingFaceTB/SmolLM2-135M-Instruct | 135M | 0.27 GB | 1.8 s | 1.1 s | 0.7 s |
| Qwen/Qwen2.5-0.5B-Instruct | 494M | 0.99 GB | 2.1 s | 3.3 s | 0.7 s |
| Qwen/Qwen2.5-1.5B-Instruct | 1544M | 3.09 GB | 1.4 s | 9.9 s | 1.1 s |

## Results 1: sanity test (all reflexive methods x 3 models)

10 easy questions, each with the correct and a deliberately wrong answer (20 examples), measured with R2's shared
`run_sanity()`. AUROC with correct answer = positive class: >= 0.8 works, 0.6-0.8 weak signal, < 0.6 random.

In [5]:
SHORT = {"SmolLM2-135M-Instruct": "SmolLM2-135M", "Qwen2.5-0.5B-Instruct": "Qwen-0.5B", "Qwen2.5-1.5B-Instruct": "Qwen-1.5B"}
ORDER = [("p_true_zeroshot_norm", "R1"), ("p_true_zeroshot_raw", "R1"), ("r2_letter_nobrainstorm", "R2"),
         ("r2_letter_brainstorm", "R2"), ("p_true_brainstorm", "R2"), ("p_true_empirical", "R3"),
         ("verbalized_2s", "R3"), ("verbalized_2s_10", "R3")]

sanity = {}
for f in sorted((ROOT / "results/sanity").glob("*.json")):
    d = json.loads(f.read_text())
    short = SHORT.get(d["model"].split("/")[-1])
    if short:
        sanity[(d["method"], short)] = d

owner = dict(ORDER)
methods = [m for m, _ in ORDER if any((m, s) in sanity for s in SHORT.values())]
methods += sorted({m for m, _ in sanity} - set(methods))
lines = ["| Method | Who | " + " | ".join(SHORT.values()) + " |", "|---|---|---|---|---|"]
for m in methods:
    cells = [f"{sanity[(m, s)]['auroc']:.2f}" if (m, s) in sanity else "-" for s in SHORT.values()]
    lines.append(f"| {m} | {owner.get(m, '?')} | " + " | ".join(cells) + " |")
display(Markdown("**Sanity AUROC** (- = not run yet)\n\n" + "\n".join(lines)))

**Sanity AUROC** (- = not run yet)

| Method | Who | SmolLM2-135M | Qwen-0.5B | Qwen-1.5B |
|---|---|---|---|---|
| p_true_zeroshot_norm | R1 | 0.40 | 0.70 | 0.97 |
| p_true_zeroshot_raw | R1 | 0.47 | 0.57 | 0.91 |
| r2_letter_nobrainstorm | R2 | 0.37 | 0.76 | 0.99 |
| r2_letter_brainstorm | R2 | 0.34 | 1.00 | 1.00 |
| p_true_empirical | R3 | 0.52 | 0.58 | 0.94 |
| verbalized_2s | R3 | 0.45 | 0.54 | 0.60 |
| verbalized_2s_10 | R3 | 0.50 | 0.45 | 0.59 |

In [6]:
# R3 details: does the method produce different numbers at all, and how often can the reply not be read?
lines = ["| Method | Model | AUROC | Gap | n_distinct | Failed / unreadable | s per example |", "|---|---|---|---|---|---|---|"]
for m in ["p_true_empirical", "verbalized_2s", "verbalized_2s_10"]:
    for s in SHORT.values():
        d = sanity.get((m, s))
        if d is None:
            continue
        extra = f"{int(d['failed'])}/20 failed" if "failed" in d else f"{d['unparsed_share']:.0%} unreadable"
        lines.append(f"| {m} | {s} | {d['auroc']:.2f} | {d['gap']:+.2f} | {d['n_distinct']} | {extra} | {d['seconds_per_example']:.2f} |")
display(Markdown("\n".join(lines)))

| Method | Model | AUROC | Gap | n_distinct | Failed / unreadable | s per example |
|---|---|---|---|---|---|---|
| p_true_empirical | SmolLM2-135M | 0.52 | -0.02 | 7 | 42% unreadable | 0.16 |
| p_true_empirical | Qwen-0.5B | 0.58 | +0.06 | 7 | 8% unreadable | 0.39 |
| p_true_empirical | Qwen-1.5B | 0.94 | +0.88 | 4 | 0% unreadable | 1.23 |
| verbalized_2s | SmolLM2-135M | 0.45 | -0.10 | 6 | 0/20 failed | 0.08 |
| verbalized_2s | Qwen-0.5B | 0.54 | +0.01 | 3 | 0/20 failed | 0.08 |
| verbalized_2s | Qwen-1.5B | 0.60 | +0.01 | 2 | 0/20 failed | 0.20 |
| verbalized_2s_10 | SmolLM2-135M | 0.50 | +0.00 | 2 | 0/20 failed | 0.08 |
| verbalized_2s_10 | Qwen-0.5B | 0.45 | -0.01 | 2 | 0/20 failed | 0.06 |
| verbalized_2s_10 | Qwen-1.5B | 0.59 | +0.02 | 3 | 0/20 failed | 0.15 |

In [7]:
# What the replies look like: the wrong answer "Berlin" per model, and the verbalized numbers on Qwen-1.5B
for s in SHORT.values():
    d = sanity.get(("p_true_empirical", s))
    if d:
        ex = next(e for e in d["examples"] if e["a"] == "Berlin")
        print(f"{s:<13} Berlin -> p(True) = {ex['conf']:.1f}   {ex['raw'][:4]} ...")
d = sanity.get(("verbalized_2s", "Qwen-1.5B"))
if d:
    print("\nQwen-1.5B verbalized replies:", sorted({e["raw"] for e in d["examples"]}))

SmolLM2-135M  Berlin -> p(True) = 0.0   ['(B)', 'Bering Island', 'Yes, the', 'The capital of'] ...
Qwen-0.5B     Berlin -> p(True) = 0.9   ['(A) True', '(A) True', '(A) True', '(A) True'] ...
Qwen-1.5B     Berlin -> p(True) = 0.0   ['(B) False', '(B) False', '(B) False', '(B) False'] ...

Qwen-1.5B verbalized replies: ['100', '95']


## Results 2: 50 TriviaQA questions (Qwen2.5-0.5B)

Each method scores the model's own greedy answer; uncertainty = 1 - confidence (score contract). AUROC with
wrong answer = positive class; AURC is lower = better, and a random ranking gives AURC ~ error rate. 95 %
bootstrap interval over the questions. R1's zero-shot p(True) is shown for comparison.

In [8]:
try:  # B2's shared evaluation (common/evaluate.py, common/io.py)
    from common.evaluate import aurc, auroc, bootstrap_auroc
    from common.io import load_scores
    SOURCE = "B2 common.evaluate"
except ImportError:  # until B2's code is on this branch
    from r3_real_data import aurc
    from sklearn.metrics import roc_auc_score

    def auroc(u, c):
        return float(roc_auc_score(~np.asarray(c, dtype=bool), u))

    def bootstrap_auroc(u, c, n_boot=1000, seed=0):
        u, c, rng, vals = np.asarray(u), np.asarray(c, dtype=bool), np.random.default_rng(seed), []
        for _ in range(n_boot):
            i = rng.integers(0, len(u), len(u))
            if 0 < c[i].sum() < len(u):
                vals.append(auroc(u[i], c[i]))
        return tuple(np.percentile(vals, [2.5, 97.5]))

    def load_scores(path, expected_ids=None):
        return json.loads(Path(path).read_text())

    SOURCE = "R3 fallback (B2's evaluate not on this branch yet)"

lines = ["| Method | Who | AUROC | 95% CI | AURC | Failed | n_distinct | s / question |", "|---|---|---|---|---|---|---|---|"]
for m, who in [("p_true_zeroshot_norm", "R1"), ("p_true_brainstorm", "R2"), ("p_true_empirical", "R3"), ("verbalized_2s", "R3")]:
    f = ROOT / f"results/qwen0.5b/{m}.json"
    if not f.exists():
        lines.append(f"| {m} | {who} | - | - | - | - | - | not run yet |")
        continue
    rec = load_scores(f, expected_ids=ids)
    u = np.asarray(rec["uncertainty"])
    lo, hi = bootstrap_auroc(u, correct)
    lines.append(f"| {m} | {who} | {auroc(u, correct):.2f} | {lo:.2f}-{hi:.2f} | {aurc(u, correct):.3f} | "
                 f"{sum(rec['failed'])}/50 | {len(np.unique(np.round(u, 3)))} | {rec['seconds_per_question']:.2f} |")
display(Markdown(f"Evaluation: {SOURCE}. Random AURC ~ {1 - correct.mean():.2f}\n\n" + "\n".join(lines)))

Evaluation: B2 common.evaluate. Random AURC ~ 0.76

| Method | Who | AUROC | 95% CI | AURC | Failed | n_distinct | s / question |
|---|---|---|---|---|---|---|---|
| p_true_zeroshot_norm | R1 | 0.65 | 0.45-0.83 | 0.672 | 0/50 | 46 | 0.79 |
| p_true_brainstorm | R2 | - | - | - | - | - | not run yet |
| p_true_empirical | R3 | 0.53 | 0.36-0.71 | 0.746 | 0/50 | 8 | 0.50 |
| verbalized_2s | R3 | 0.43 | 0.27-0.61 | 0.793 | 0/50 | 4 | 0.11 |

In [9]:
# Why it does not separate: confidence on correct vs wrong answers, and which values occur
from collections import Counter

for m in ["p_true_empirical", "verbalized_2s"]:
    det = json.loads((ROOT / f"results/qwen0.5b/{m}_details.json").read_text())
    c = [e["confidence"] for e in det if e["correct"]]
    w = [e["confidence"] for e in det if not e["correct"]]
    print(f"{m}: mean confidence correct {np.mean(c):.2f} | wrong {np.mean(w):.2f}")
    print("   values:", dict(sorted(Counter(round(e["confidence"], 2) for e in det).items())))

p_true_empirical: mean confidence correct 0.86 | wrong 0.83
   values: {0.3: 1, 0.4: 1, 0.5: 4, 0.6: 4, 0.7: 7, 0.8: 4, 0.9: 5, 1.0: 24}
verbalized_2s: mean confidence correct 0.89 | wrong 0.91
   values: {0.8: 2, 0.85: 20, 0.9: 2, 0.95: 26}


## Results 3: cost per question

Model calls and time per question on Qwen2.5-0.5B (`scripts/r3_cost_table.py`).

In [10]:
cost = ROOT / "results/qwen0.5b/cost_table.md"
display(Markdown(cost.read_text().split("\n", 2)[2] if cost.exists() else "cost_table.md missing: run scripts/r3_cost_table.py"))

| Method | Who | Model calls / question | Generated tokens | Needs logits? | s / question (Qwen 0.5B) |
|---|---|---|---|---|---|
| p_true_zeroshot_norm | R1 | 1 forward pass | 0 | yes | 0.79 |
| p_true_brainstorm | R2 | 1 forward pass + 10 samples* | 0 | yes | pending |
| p_true_empirical | R3 | 10 generations | <= 3 x 10 | no | 0.50 |
| verbalized_2s | R3 | 1 generation | <= 8 | no | 0.11 |

*Brainstorm reuses the 10 samples in B1's data file; they are generated for semantic entropy anyway, so they add no extra cost here. Run from scratch: +10 generations.
Methods without logits (R3) only see text, so they also work with API models.


## What works / What does not

**Works**
- Both methods run on text alone (no logits), so they carry over to API models.
- Empirical p(True) works from 1.5B on: sanity AUROC 0.94 (Paris 10/10 True, Berlin 10/10 False), close to
  R1's logit-based zero-shot p(True) (0.97).
- On the 50 real questions R1's logit-based p(True) does somewhat better (AUROC 0.65), but its interval also
  includes 0.5.
- Results follow the shared score contract and pass B2's validation.

**Does not work**
- On Qwen-0.5B both methods are at chance level on 50 real questions (AUROC 0.53 and 0.43; the bootstrap
  intervals include 0.5). The model says "True" to almost everything (24 of 50 questions: 10/10 True).
- Verbalized confidence does not discriminate at any size: the models answer 85, 90, 95 or 100 for almost
  every answer (2-4 distinct values). A 0-10 scale (the plan's fallback) did not help.
- SmolLM2-135M often does not follow the True/False format (42 % unreadable replies).

**Cost:** empirical p(True) needs 10 generations per question (~0.5 s), verbalized 1 (~0.1 s), zero-shot
p(True) 1 forward pass (~0.8 s).

*Caveat:* only 12 of the 50 greedy answers are correct, so the 50-question numbers are noisy.

## Questions for tutors

1. Should cost (model calls per question) be part of the comparison?
2. Reflexive methods only start working at 1.5B. Should the main comparison use Qwen2.5-1.5B instead of 0.5B?
3. Empirical p(True): keep temperature 1.0 and 10 samples, or try more samples?